# Utility-Preserving Safety Analysis

**Objective.** Audit the claim that ORIUS is not merely a shutdown, always-brake, or always-alert policy.

**Run mode.** Analysis only. This notebook reads locked ORIUS artifacts
and does not retrain models, rewrite release manifests, or mutate runtime traces.

In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "reports").exists():
    ROOT = Path.cwd().parent
PUBLICATION = ROOT / "reports" / "publication"
SPLIT_ROOT = ROOT / "reports" / "split_training"
RELEASE_ID = (SPLIT_ROOT / "latest_release_id.txt").read_text().strip()
FREEZE = ROOT / "reports" / "predeployment_freeze" / RELEASE_ID

def read_csv(relpath: str) -> pd.DataFrame:
    path = ROOT / relpath
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

def read_json(relpath: str) -> dict:
    path = ROOT / relpath
    if not path.exists():
        raise FileNotFoundError(path)
    return json.loads(path.read_text())

def display_path(relpath: str) -> None:
    path = ROOT / relpath
    print(f"{relpath}: {'exists' if path.exists() else 'missing'}")

## Scorecard semantics

In [ ]:
scorecard = read_csv("reports/publication/utility_preserving_safety_scorecard.csv")
cols = [
    "domain",
    "safety_reference_controller",
    "orius_tsvr",
    "safety_reference_tsvr",
    "excess_tsvr_over_safety_reference",
    "orius_utility",
    "safety_reference_utility",
    "utility_gain_over_safety_reference",
    "utility_preserving_safety_gate",
    "claim_boundary",
]
scorecard[cols]

In [ ]:
plot_df = scorecard.copy()
plot_df["utility_delta_over_safety_reference"] = pd.to_numeric(
    plot_df["utility_delta_over_safety_reference"], errors="coerce"
)
ax = plot_df.plot(
    x="domain",
    y="utility_delta_over_safety_reference",
    kind="bar",
    legend=False,
    figsize=(9, 4),
    title="Useful work preserved over fail-safe reference",
)
ax.set_ylabel("Utility delta")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()

In [ ]:
assert (scorecard["excess_tsvr_over_safety_reference"].astype(float) <= 1e-3).all()
assert (scorecard["utility_preserving_safety_gate"].astype(str).str.lower() == "true").all()